# Using coords2sans

This notebook computes small-angle neutron scattering (SANS) curves
I(q) from the atom coordinates of a CTAB rod micelle, in two ways:

1. **One curve, blocking:** read one structure and compute its curve
   in this process.
2. **Many curves side by side:** several files and contrasts, each
   computed in its own worker process.

The files in `data/` hold the 178 CTA molecules of one micelle after
the first (eq0) and the last (eq5) equilibration stage of a GROMACS
run. Water and bromide ions were removed to keep the files small; the
calculation leaves them out anyway.

In [ ]:
import tempfile
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

from coords2sans import SansCalculator
from coords2sans import SansTask
from coords2sans import Structure

## Atom masses

Every atom needs its element, for its scattering length and volume.
The safest source is the atom masses of each residue, in the order of
its topology; coords2sans matches each mass to the nearest element.
In your own work they come from your topology reader. Here they are
typed in from `CTA.itp`: N1, three methyl groups, 15 CH2 groups and
one end methyl.

In [ ]:
N, C, H = 14.007, 12.011, 1.008
cta_masses = [N] + [C, H, H, H] * 3 + [C, H, H] * 15 + [C, H, H, H]
len(cta_masses)

## 1. One curve, blocking

Example setup: the micelle after the last equilibration stage,
measured in D2O, with hydrogenated CTA. The micelles of the sample
differ in size by about 10 %.

Each comment gives the range in which you choose the value; this
example uses one value from it.

In [ ]:
# any .gro or .pdb: a sample model, an equilibration stage,
# a frame from any MD program
structure_file = Path('data/CTA178-eq5.gro')
# residue names of the aggregate: one surfactant up to every lipid
# of a mixed vesicle; the other residues (water, ions) are solvent
aggregate_residues = ['CTA']
# atom masses per residue, in topology order: {} (elements from
# other sources) up to every residue of the aggregate (safest)
residue_masses = {'CTA': cta_masses}
# residues whose hydrogens are deuterium: () (all hydrogenated)
# up to every residue of the aggregate
deuterated_residues = ()
# solvent SLD in Å⁻²: -0.56e-6 (H2O) to 6.36e-6 (D2O);
# H2O/D2O mixtures lie on the line between
solvent_sld = 6.36e-6
# q range in Å⁻¹: SANS instruments cover about 0.001 to 0.6;
# points: about 100 to 1000
q_min, q_max, q_points = 0.002, 0.2, 256
# q spacing: True (log, for log-log plots and wide ranges)
# or False (linear)
q_log = True
# volume fraction of the aggregates: above 0, up to 1;
# dilute samples about 0.001 to 0.05
scale = 0.01
# flat background in cm⁻¹: 0 (none) up to about 1 for samples
# rich in H2O; about 0.05 in D2O
background = 0.05
# spread of the micelle sizes: 0 (one size) to about 0.3;
# well-defined micelles about 0.05 to 0.15
polydispersity = 0.1
# sizes computed for the spread: about 20 to 50; double it and
# keep it if the curve stays the same
size_points = 35
# averaging over directions: False (exact, over all atom pairs;
# about 2 s here) or True (SasView's is_avg: much faster above
# 100 000 atoms, but right only for round aggregates)
average_amplitude = False

`Structure.read` keeps only the aggregate's residues, puts molecules
split by the periodic box back together, and finds each atom's
element. `element_sources` says where the elements came from; check
it before you trust a curve, because `'names'` means a guess.

In [ ]:
structure = Structure.read(structure_file, residues=aggregate_residues, masses=residue_masses)
structure.element_sources

In [ ]:
calculator = SansCalculator(
    solvent_sld,
    deuterated=deuterated_residues,
    q_min=q_min,
    q_max=q_max,
    q_points=q_points,
    q_log=q_log,
    scale=scale,
    background=background,
    polydispersity=polydispersity,
    size_points=size_points,
    average_amplitude=average_amplitude,
)
curve = calculator.calculate(structure, label='eq5 in D2O')
curve.plot();

`curve.save` writes two columns, q in Å⁻¹ and I in cm⁻¹, that
SasView opens as they are. Here they go to a temporary folder.

In [ ]:
output_dir = Path(tempfile.mkdtemp())  # any folder you like
curve.save(output_dir / 'eq5-D2O.dat')
sorted(path.name for path in output_dir.iterdir())

### What polydispersity does

The settings are plain attributes: change one and calculate again.
A wider spread of sizes smooths the minima of the curve. The curve
also rises at low q, because big micelles scatter much more than
small ones. All sizes share one pass over the atom pairs; each size
adds only a short sum.

In [ ]:
ax = None
for spread in (0.0, 0.1, 0.2):
    calculator.polydispersity = spread
    ax = calculator.calculate(structure, f'polydispersity {spread}').plot(ax)

## 2. Many curves side by side

Example setup: the micelle after the first and the last
equilibration stage, each in H2O and in D2O: four curves, one size
each, to see how the curve changes while the micelle equilibrates.

A `SansTask` holds a file path and a calculator, not arrays, so
sending it to a worker process is cheap. On Windows and macOS,
workers start fresh ("spawn"): they can run `SansTask.run` because
it lives in the package. A function defined in this notebook would
not work there.

In [ ]:
# structure files: one file up to every stage or trajectory frame
stage_files = [Path('data/CTA178-eq0.gro'), Path('data/CTA178-eq5.gro')]
# solvent SLD in Å⁻² per contrast: one or more between -0.56e-6
# (H2O) and 6.36e-6 (D2O)
contrasts = {'H2O': -0.56e-6, 'D2O': 6.36e-6}
# worker processes: 1 up to the number of CPU cores;
# each holds one structure in memory
workers = 4

tasks = [
    SansTask(
        structure_file=stage_file,
        calculator=SansCalculator(solvent_sld, scale=scale),
        residues=aggregate_residues,
        masses=residue_masses,
        label=f'{stage_file.stem} in {contrast}',
    )
    for stage_file in stage_files
    for contrast, solvent_sld in contrasts.items()
]
with ProcessPoolExecutor(max_workers=workers) as pool:
    curves = list(pool.map(SansTask.run, tasks))

ax = None
for curve in curves:
    ax = curve.plot(ax)

`task.run()` also works without a pool, in this process. The
`compare` notebook checks these curves against SasView and
sasmodels.